# POLITSKY24 Dataset Accessing and Testing

In [1]:
import os
import pandas as pd

from pathlib import Path
from dotenv import load_dotenv

# Load environment variables from .env file
load_dotenv()

# Change path on .env file on your local machine
data_path = Path(os.getenv("DATA_PATH"))

In [2]:
# Access the parquet files in the data path

parquet_files = list(data_path.glob("*.parquet"))
parquet_files

[WindowsPath('D:/Thesis-related/POLITISKY24/human_annotated_validation_stance_relevancy_dataset (post-target entity pairs).parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/human_annotated_validation_stance_relevancy_dataset (query-post stance relevancy pairs).parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/human_annotated_validation_user_stance_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/llm_annotated_full_user_stance_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/llm_annotated_validation_user_stance_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/user_network_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/user_post_history_dataset.parquet'),
 WindowsPath('D:/Thesis-related/POLITISKY24/user_post_list_for_stance_detection.parquet')]

In [3]:
import polars as pl

# Load each dataset manually into individual variables
user_network_df = pl.read_parquet(data_path / "user_network_dataset.parquet")
user_post_history_df = pl.read_parquet(data_path / "user_post_history_dataset.parquet")
user_post_list_df = pl.read_parquet(data_path / "user_post_list_for_stance_detection.parquet")

# Stance label datasets
llm_full_stance_df = pl.read_parquet(data_path / "llm_annotated_full_user_stance_dataset.parquet")
llm_val_stance_df = pl.read_parquet(data_path / "llm_annotated_validation_user_stance_dataset.parquet")
human_val_stance_df = pl.read_parquet(data_path / "human_annotated_validation_user_stance_dataset.parquet")

# Stance relevancy datasets (Note: surround filename strings in quotes)
human_val_relevancy_post_df = pl.read_parquet(
    data_path / "human_annotated_validation_stance_relevancy_dataset (post-target entity pairs).parquet"
)
human_val_relevancy_query_df = pl.read_parquet(
    data_path / "human_annotated_validation_stance_relevancy_dataset (query-post stance relevancy pairs).parquet"
)

In [4]:
user_network_df.schema

Schema([('SourceUserId', Int64),
        ('TargetUserId', Int64),
        ('InteractionType', String),
        ('Count', Int64)])

In [13]:
user_network_df.head()

SourceUserId,TargetUserId,InteractionType,Count
i64,i64,str,i64
9,9,"""Like""",1
9,9,"""Quote""",42
9,9,"""Repost""",4
9,10,"""Quote""",2
9,2549,"""Quote""",1


In [22]:
# Check number of rows in user_network_df

user_network_df.shape[0]

1676895

In [5]:
user_post_history_df.schema

Schema([('PostId', Int64),
        ('UserId', Int64),
        ('PostTime', String),
        ('Content', String),
        ('Hashtags', List(String)),
        ('Languages', List(String)),
        ('Mentions', List(Int64)),
        ('IsRepost', Boolean),
        ('SourceUserId', Int64),
        ('IsQuote', Boolean),
        ('QuoteId', Int64),
        ('QuoteUserId', Int64),
        ('QuoteContent', String),
        ('IsReply', Boolean),
        ('ParentId', Int64),
        ('ParentUserId', Int64)])

In [6]:
user_post_history_df.select(pl.all().n_unique()).to_dicts()[0]

{'PostId': 14830231,
 'UserId': 8561,
 'PostTime': 18230517,
 'Content': 13138778,
 'Hashtags': 285770,
 'Languages': 1193,
 'Mentions': 88178,
 'IsRepost': 2,
 'SourceUserId': 217858,
 'IsQuote': 2,
 'QuoteId': 975337,
 'QuoteUserId': 105488,
 'QuoteContent': 910916,
 'IsReply': 2,
 'ParentId': 5472655,
 'ParentUserId': 291653}

In [8]:
# Exact duplicate rows (list columns may need to be dropped/converted first)
user_post_history_df.is_duplicated().sum()

# Rows sharing a PostId
dupes = user_post_history_df.filter(pl.col("PostId").is_duplicated())
dupes.sort("PostId").head(20)

PostId,UserId,PostTime,Content,Hashtags,Languages,Mentions,IsRepost,SourceUserId,IsQuote,QuoteId,QuoteUserId,QuoteContent,IsReply,ParentId,ParentUserId
i64,i64,str,str,list[str],list[str],list[i64],bool,i64,bool,i64,i64,str,bool,i64,i64
18,16171,"""2023-12-03T01:11:35.122Z""","""The end of Moorevember coincid…",[],"[""en""]",[],true,7,false,-1,-1,"""""",false,-1,-1
18,71390,"""2023-12-04T22:26:06.295Z""","""The end of Moorevember coincid…",[],"[""en""]",[],true,7,false,-1,-1,"""""",false,-1,-1
47,9,"""2023-06-22T00:48:37.165Z""","""I used a Chrome/Edge extension…",[],[],[],false,9,false,-1,-1,"""""",false,15691770,343645
47,285903,"""2023-06-22T00:49:34.781Z""","""I used a Chrome/Edge extension…",[],[],[],true,9,false,-1,-1,"""""",false,15691770,343645
279,9,"""2023-07-06T17:04:05.906Z""","""Should be required that PD off…",[],"[""en""]",[],false,9,false,-1,-1,"""""",false,4244240,90714
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
999,141415,"""2023-10-03T01:52:13.292Z""","""Let's talk about how domestic …",[],"[""en""]",[],true,9,false,-1,-1,"""""",false,-1,-1
1000,90924,"""2023-10-03T01:47:26.282Z""","""Going along with Sheryl's post…",[],"[""en""]",[],true,9,false,-1,-1,"""""",true,999,9
1000,9,"""2023-10-03T00:58:13.600Z""","""Going along with Sheryl's post…",[],"[""en""]",[],false,9,false,-1,-1,"""""",true,999,9


In [ ]:
user_post_history_df.shape

(18416787, 16)

In [ ]:
user_post_list_df.schema

Schema([('UserId', Int64), ('PostIdList', List(Int64))])

In [14]:
llm_full_stance_df.schema

Schema([('UserId', Int64),
        ('TargetEntity', String),
        ('Context', List(String)),
        ('LLMAnswerContent', String),
        ('LLMAnswerUsage', String),
        ('SourcePosts', List(String)),
        ('Spans', List(String)),
        ('Reason', String),
        ('StanceLabel', String),
        ('ConfidenceLevel', Float64)])

In [16]:
# Get number of unique values in the stance labbel column
llm_full_stance_df.select(pl.col("StanceLabel").n_unique()).to_dicts

<bound method DataFrame.to_dicts of shape: (1, 1)
┌─────────────┐
│ StanceLabel │
│ ---         │
│ u32         │
╞═════════════╡
│ 3           │
└─────────────┘>

In [21]:
# Get the count of each stance label using the value_counts() method
stance_counts = llm_full_stance_df["StanceLabel"].value_counts()
stance_counts

StanceLabel,count
str,u32
"""Favor""",3071
"""Neither""",5141
"""Against""",7832


In [17]:
llm_full_stance_df.head()

UserId,TargetEntity,Context,LLMAnswerContent,LLMAnswerUsage,SourcePosts,Spans,Reason,StanceLabel,ConfidenceLevel
i64,str,list[str],str,str,list[str],list[str],str,str,f64
9,"""Harris""","[""Assuming Harris is the pick. Biden might have endorsed her but I sure as shit hope there's been arrangements to make her the nominee with no fuss."", ""I mean so do I, there's several people I'd want over Harris. That's...not the same as refusing to vote for or support Harris in THIS election. Someone criticizing Harris for specific positions does not automatically equal ""will not vote for her"""", … ""Jill Stein is trash and cannot be trusted. https://anonymous""]","""{ ""source_tweets"": [ …","""{'input_tokens': 888, 'output_…","[""Assuming Harris is the pick. Biden might have endorsed her but I sure as shit hope there's been arrangements to make her the nominee with no fuss."", ""I mean so do I, there's several people I'd want over Harris. That's...not the same as refusing to vote for or support Harris in THIS election."", … ""Cannot express how badly this whole 'Dump Biden' thing could go if it also pulls in Harris""]","[""hope there's been arrangements to make her the nominee with no fuss"", ""not the same as refusing to vote for or support Harris in THIS election"", … ""Cannot express how badly this whole 'Dump Biden' thing could go if it also pulls in Harris""]","""The user expresses a preferenc…","""Favor""",0.85
9,"""Trump""","[""Gonna fucking Battle Royale us into a Trump presidency"", ""I don't see how you read ""We are withholding our endorsement out of fear that, should he win, Donald Trump will retaliate against our newspaper"" as anything but a giant ""VOTE KAMALA HARRIS!!!!!!!!"" statement."", … ""Hell I'm still seeing BRUCE SPRINGSTEEN takes that are ""too bad he went woke and liberal!""""]","""{ ""source_tweets"": [ …","""{'input_tokens': 810, 'output_…","[""Gonna fucking Battle Royale us into a Trump presidency"", ""I don't see how you read ""We are withholding our endorsement out of fear that, should he win, Donald Trump will retaliate against our newspaper"" as anything but a giant ""VOTE KAMALA HARRIS!!!!!!!!"" statement."", … ""Any undecided voter who's paid even the slightest attention is a Trump voter trying to justify their vote.""]","[""Gonna fucking Battle Royale us into a Trump presidency"", ""We are withholding our endorsement out of fear that, should he win, Donald Trump will retaliate against our newspaper"", … ""Any undecided voter who's paid even the slightest attention is a Trump voter trying to justify their vote""]","""The tweets express a critical …","""Against""",0.95
10,"""Harris""","[""Kamala/Harris"", ""Kamala Harris is very good at this."", … ""Republicans attacks against Kamala thus far: - She's hot - She enjoys laughing - She's quirky""]","""{ ""source_tweets"": [ …","""{'input_tokens': 803, 'output_…","[""Kamala Harris is very good at this."", ""I think Kamala is going to win."", … ""They need to run against Elon Musk, Peter Thiel, David Sacks and the rest of these white nationalist billionaires who oppose unions, democracy, and anything which gives working people a say.""]","[""Kamala Harris is very good at this."", ""I think Kamala is going to win."", … ""run against Elon Musk, Peter Thiel, David Sacks and the rest of these white nationalist billionaires""]","""The tweets express strong supp…","""Favor""",0.95
10,"""Trump""","[""Not to get too political on here, but I really hope Donald Trump does not become president again."", ""Nothing but respect for my president 🥥🌴🥥🌴"", … ""AP still has Trump up""]","""{ ""source_tweets"": [""Not t…","""{'input_tokens': 808, 'output_…","[""Not to get too political on here, but I really hope Donald Trump does not become president again."", ""Not to get too political on here, but I really hope that Donald Trump does not win the presidential election."", ""If Donald Trump wins the presidential election after going full-fascist, I think that will be quite bad

In [ ]:
llm_val_stance_df.schema


Schema([('UserId', Int64),
        ('TargetEntity', String),
        ('Context', List(String)),
        ('LLMAnswerContent', String),
        ('LLMAnswerUsage', String),
        ('SourcePosts', List(String)),
        ('Spans', List(String)),
        ('Reason', String),
        ('StanceLabel', String),
        ('ConfidenceLevel', Float64)])

In [ ]:
human_val_stance_df.schema

Schema([('UserId', Int64), ('Trump', String), ('Harris', String)])

In [ ]:
human_val_relevancy_post_df.schema

Schema([('PostId', Int64), ('Trump', Int64), ('Harris', Int64)])

In [ ]:
human_val_relevancy_query_df.schema

Schema([('Query', String),
        ('PostId', Int64),
        ('StanceRelevanceLabel', String)])

## Graph Creation Testing

The code under this section of the notebook is untested and unverified. Double check before being tested.

Simple idea of how we create the data needed for the GNN: User Network Graph creation (likes and reposts) --> Connect this to the LLM full stance using the user id. <br>
    - Verify if they have the same id types (int64) and if they can be connect to each other

In [ ]:
import polars as pl

core = user_post_history_df["UserId"].unique()

reposts = (
    user_post_history_df
    .filter(pl.col("IsRepost"))
    .filter(pl.col("SourceUserId").is_in(core))       # keep core users only
    .filter(pl.col("UserId") != pl.col("SourceUserId"))  # drop self-loops
    .group_by("UserId", "SourceUserId")
    .agg(pl.len().alias("weight"))
    .rename({"UserId": "src", "SourceUserId": "dst"})
)

In [ ]:
net = pl.read_csv("user_network.csv")  # use the actual filename/format from Zenodo
likes = (
    net.filter(pl.col("InteractionType") == "Like")  # adjust column names to match the file
    .select(src="SourceUserId", dst="TargetUserId", weight="Count")
)

In [ ]:
import torch

nodes = core.sort()
idx = {u: i for i, u in enumerate(nodes.to_list())}

def to_edge_index(edges):
    s = [idx[u] for u in edges["src"]]
    d = [idx[u] for u in edges["dst"]]
    return torch.tensor([s, d]), torch.tensor(edges["weight"].to_list(), dtype=torch.float)

edge_index, edge_weight = to_edge_index(likes)  # or reposts